# Assembly Pairs Generator
Generate mesh pairs from assembly dataset by deactivating "is_visible" from assembly.json file.

**You need to download the dataset first**

In [ ]:
import os
import shutil
import trimesh
from pathlib import Path
import numpy as np
import meshplot as mp
from IPython.display import HTML, display
from assembly_graph import AssemblyGraph
from assembly_deactivate_vis import is_visible_toggle


## Parse the Assembly

In [ ]:
assembly_dir = ""  # TODO: Enter path to assembly dataset
save_dir = ""  # TODO: Enter path where you want to store the adjusted assembly object
assembly_name = "52655_ec869bd7"  # TODO: Enter object_id of the object you want to adjust

assembly_file = os.path.join(assembly_dir, assembly_name, "assembly.json")
ag = AssemblyGraph(assembly_file)
print(f"Loaded: {assembly_file}")
graph = ag.get_graph_networkx()

## View the Assembly

In [4]:
vertices_list = []
faces_list = []
face_offset = 0
for index, (node_key, node_data) in enumerate(graph.nodes.data()):
    node_obj_file = assembly_file.parent / f"{node_data['body_file']}.obj"
    mesh = trimesh.load(node_obj_file, force='mesh')
    f = mesh.faces
    v = mesh.vertices
    faces_list.append(f + face_offset)
    v = np.pad(v.T, ((0, 1), (0, 0)), mode="constant", constant_values=1)
    transform = np.array(node_data["transform"])
    v = transform @ v
    vertices_list.append(v.T)
    face_offset += v.shape[1]
    
vertices = np.concatenate(vertices_list)
faces = np.concatenate(faces_list)
mp.plot(vertices[:,0:3], faces)

mesh_org = trimesh.Trimesh(vertices=vertices[:,0:3], faces=faces, process=False)
rotation_matrix = trimesh.transformations.rotation_matrix(
    angle=np.radians(-90),
    direction=[1, 0, 0],
    point=[0, 0, 0]
)
# mesh_org.apply_transform(rotation_matrix)
ivt = is_visible_toggle(assembly_file)
ivt.reset_visibility()
visible_count, visible_ids = ivt.get_node_visibility()
print(f"Visible count: {visible_count}, Visible IDs: {visible_ids}")


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(105.50926…

Visible count: 17, Visible IDs: {0: 'afbaca8c-0527-11ec-808d-0adcfe058e47', 1: 'afc48e74-0527-11ec-b95a-0adcfe058e47', 2: 'afc4dcb8-0527-11ec-b4a8-0adcfe058e47', 3: 'afc503fa-0527-11ec-8749-0adcfe058e47', 4: 'afc52ae2-0527-11ec-8628-0adcfe058e47', 5: 'afc57924-0527-11ec-9c26-0adcfe058e47', 6: 'afc689fe-0527-11ec-a92e-0adcfe058e47', 7: 'afc6d818-0527-11ec-8f50-0adcfe058e47', 8: 'afc77428-0527-11ec-b5e9-0adcfe058e47', 9: 'afc7e96c-0527-11ec-a1fb-0adcfe058e47', 10: 'afc81068-0527-11ec-b82c-0adcfe058e47', 11: 'afc85e8a-0527-11ec-9dc0-0adcfe058e47', 12: 'afc8acac-0527-11ec-854a-0adcfe058e47', 13: 'afc8d3b6-0527-11ec-9598-0adcfe058e47', 14: 'afc921d8-0527-11ec-b3e2-0adcfe058e47', 15: 'afc96ff8-0527-11ec-a47b-0adcfe058e47', 16: 'afb9b91e-0527-11ec-a389-0adcfe058e47'}


In [ ]:
html_table = """
<table>
  <tr>
    <th>Index</th>
    <th>ID</th>
    <th>Image</th>
  </tr>
"""

assembly_image_path = os.path.join(assembly_file.parent, "assembly.png")
html_table += f"""
<tr>
    <td>assembly</td>
    <td>{assembly_name}</td>
    <td><img src="{assembly_image_path}" width="200"></td>
</tr>
"""

for index, id in visible_ids.items():
    image_path = os.path.join(assembly_file.parent, f"{id}.png")    
    html_table += f"""
    <tr>
        <td>{index}</td>
        <td>{id}</td>
        <td><img src="{image_path}" width="200"></td>
    </tr>
    """

html_table += "</table>"

print(f"Visible count: {visible_count}")
display(HTML(html_table))

## Get deactivated body index

In [6]:
remove_index_list = range(len(visible_ids))
# vis_index_list = [0]
index_list = [16] #+ list(range(19, 22)) #+ list(range(20,31))
if 'vis_index_list' in locals():
    result_list = [item for item in remove_index_list if item not in vis_index_list]
else:
    result_list = index_list

body_ids_to_toggle = []
for i in result_list:
    body_ids_to_toggle.append(visible_ids[i])
print(f"Body IDs to toggle: {body_ids_to_toggle}")

Body IDs to toggle: ['afb9b91e-0527-11ec-a389-0adcfe058e47']


# Export changed mesh and reset

In [ ]:
os.makedirs(save_dir + '/' + assembly_name, exist_ok=True)

mesh_org.export(save_dir + '/' +  assembly_name + '/' + "original_assembly.obj")
ag_new = AssemblyGraph(assembly_file)
graph_new = ag_new.get_graph_networkx()
vertices_list_new = []
faces_list_new = []
face_offset_new = 0
for index, (node_key, node_data) in enumerate(graph_new.nodes.data()):
    if index in result_list:
        continue
    node_obj_file = assembly_file.parent / f"{node_data['body_file']}.obj"
    print(f"Processing node {index}: {node_key}, body_file: {node_data['body_file']}")
    mesh = trimesh.load(node_obj_file, force='mesh')
    f = mesh.faces
    v = mesh.vertices
    faces_list_new.append(f + face_offset_new)
    v = np.pad(v.T, ((0, 1), (0, 0)), mode="constant", constant_values=1)
    transform = np.array(node_data["transform"])
    v = transform @ v
    vertices_list_new.append(v.T)
    face_offset_new += v.shape[1]
    
vertices_new = np.concatenate(vertices_list_new)
faces_new = np.concatenate(faces_list_new)
mp.plot(vertices_new[:,0:3], faces_new)
mesh_new = trimesh.Trimesh(vertices=vertices_new[:,0:3], faces=faces_new, process=False)
mesh_name = "assembly_new_without_" + '_'.join(map(str, index_list)) + ".obj"
print(f"Saving new mesh to {save_dir}/{assembly_name}/{mesh_name}")
mesh_new.export(save_dir + '/' +  assembly_name + '/' + mesh_name)
ivt.reset_visibility()

